# W5 (robustness variant: A–B peak-disagreement gate) — goal-progress tuning: sorted β heatmaps, and peak vs width (PFC)

Plan `docs/handoff/W5_gp_tuning_width.md`; write-up `mFC_data/code/GP_TUNING_WIDTH.md`;
fits `GLM_V3.md`. **Standalone by instruction:** nothing here modifies `glm_plots.py`,
`pfc_glm_plots.py`, `check_mirror_parity.py` or the V3 notebooks — the heatmap and every
statistic live in `gp_tuning_width.py`, which keeps its own LEC/PFC byte-identity check.

**The reading rule, decided before any figure was drawn.** `goal_progress` is time normalised by
leg duration, so a population of cells locked to a fixed *time* after (or before) reward produces
a **V-shaped width-vs-peak relation in phase space** all by itself. A positive peak–width
correlation is therefore the null expectation under time coding. Every scatter below carries the
simulated time-cell V, built from this dataset's own legs through this same code. A relation that
matches the V and vanishes in the "leans phase" stratum is time coding; a flat relation with a real
width distribution is phase coding; anything else is reported as such.

In [1]:
import os, sys, pickle, subprocess, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display

REPO = os.path.abspath('../..')
CODE = os.path.join(REPO, 'mFC_data/code')
sys.path.insert(0, CODE)
import gp_tuning_width as G
import glm_analysis_v3 as glm, w1_refit as w, run_glm_batch as rb
import pfc_glm_plots as P

# AB_GATE: keep only cells whose odd-leg and even-leg peaks are within this many bins of each
# other on the circular 90-bin axis. None = the primary analysis. A number = the ROBUSTNESS
# variant, written to its own directory; see GP_TUNING_WIDTH.md section 2.0.1 for why it is not
# the primary (the gate selects on a width-correlated variable).
AB_GATE = 30
FIGDIR = G.FIG_DIR + '_abgate'
G._ensure_writable_dir(FIGDIR)
G._style()
print('dataset', G.DATASET, '| engine', glm.GLM_VERSION, '| figures ->', FIGDIR)

# 0.1 the two copies of this module must be identical (check_mirror_parity was deliberately
# not extended -- see GP_TUNING_WIDTH.md section 0)
print('mirror OK:', [a for a, _ in G.assert_mirror()])

# 0.2 synthetic controls (refresh with `python mFC_data/code/gp_tuning_width_synthetics.py`)
SYN = pickle.load(open(G.SYNTH_OUT, 'rb'))
display(pd.DataFrame([{'control': k, 'pass': v['pass'], 'elapsed_s': v['elapsed_s']}
                      for k, v in SYN['results'].items()]))
assert all(v['pass'] for v in SYN['results'].values()), 'a synthetic control failed -- see GP_TUNING_WIDTH.md'
SIMS = SYN['sims']
if AB_GATE is not None:
    # the planted populations go through the IDENTICAL gate, or a filtered measurement would be
    # compared against an unfiltered prediction
    SIMS = SIMS[SIMS.ab_offset_bins <= AB_GATE]
    NULL = G.null_curves(SIMS[SIMS.sigma == G.SIGMA])
    print(f'A-B peak-disagreement gate ACTIVE at {AB_GATE} bins — robustness variant, '
          f'figures -> {FIGDIR}')
else:
    NULL = SYN['null_curves'][G.SIGMA]
    print('no A-B gate (primary analysis)')

dataset PFC | engine v3 | figures -> /ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data/figures/gp_tuning_width_abgate
mirror OK: ['code/gp_tuning_width.py', 'code/gp_tuning_width_synthetics.py']


,control,pass,elapsed_s
0,1 rebin == binned_statistic,True,0.0
1,2 split halves disjoint + interleaved,True,0.0
2,3 width / peak calibration (phase cells),True,0.1
3,4 time-cell null: the V and the per-state shift,True,0.2
4,5 noise cells,True,0.0
5,"6 readings separable (phase flat, time V)",True,0.0
6,7 heatmap gate: planted diagonal vs noise,True,0.1
7,8 assemble refuses a misaligned table,True,0.1


A-B peak-disagreement gate ACTIVE at 30 bins — robustness variant, figures -> /ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data/figures/gp_tuning_width_abgate


## 1. The per-leg curves

One 90-bin phase curve per leg per neuron, from the current `data_dic` (LEC) / the PFC files, over
the GLM's session set, legs longer than 30 s dropped whole. Built by
`gp_tuning_width.py --build`.

In [2]:
caches = G.load_all_caches()
rows = []
for rd, c in caches.items():
    dur = c['leg_duration_s']
    rows.append({'recday': rd, 'mouse': c['mouse'], 'sessions': len(c['sessions']),
                 'neurons': c['curves'].shape[0], 'legs': c['curves'].shape[1],
                 'legs > cap': c['n_legs_dropped_cap'], 'legs < 90 samples': c['n_legs_short'],
                 'leg s median': round(float(np.median(dur)), 2),
                 'leg s p90': round(float(np.percentile(dur, 90)), 2),
                 'rate Hz median': round(float(np.median(c['mean_rate_hz'])), 2)})
tab = pd.DataFrame(rows)
display(tab)
print(f"{len(caches)} recdays, {tab['neurons'].sum()} neurons, {tab['legs'].sum()} legs, "
      f"{tab['legs > cap'].sum()} legs over the 30 s cap, {tab['legs < 90 samples'].sum()} legs "
      f"shorter than 90 samples (upsampled, rule '{list(caches.values())[0]['short_leg_rule']}')")

,recday,mouse,sessions,neurons,legs,legs > cap,legs < 90 samples,leg s median,leg s p90,rate Hz median
0,ab03_01092023_02092023,ab03,6,63,722,10,20,8.45,15.79,5.52
1,ab03_05092023_06092023,ab03,6,64,436,40,6,10.00,19.06,5.77
2,ab03_29082023_30082023,ab03,6,70,545,11,3,9.85,17.91,4.64
3,ah03_12082021_13082021,ah03,6,16,858,14,4,5.75,11.88,3.49
4,ah03_18082021_19082021,ah03,6,15,1020,8,11,5.00,10.45,6.24
5,ah04_01122021_02122021,ah04,6,117,755,9,15,7.83,13.70,1.47
6,ah04_05122021_06122021,ah04,6,114,720,8,5,6.50,13.71,1.35
7,ah04_07122021_08122021,ah04,6,81,680,12,2,6.92,13.34,1.45
8,ah04_09122021_10122021,ah04,6,99,871,5,20,6.28,12.22,1.29
9,ah04_14122021_16122021,ah04,6,70,905,3,13,6.33,13.03,1.37


25 recdays, 1252 neurons, 15815 legs, 477 legs over the 30 s cap, 169 legs shorter than 90 samples (upsampled, rule 'rate')


### 1.1 Verification 1 — the rebuild against the legacy pickles, and currency

Two comparisons (`gp_tuning_width.py --verify`). Under the legacy short-leg convention
(`np.repeat(x, 10) / 10`, which `preprocessing/build_data_dic.normalise` used to make
`norm_neurons_dic.pkl` and, through `Smoothed_norm`, `gp_curves.pkl`) the rebuild must reproduce
those arrays **exactly**; under the rate-preserving convention this analysis uses
(`np.repeat(seg, ceil(90/L))`, as `remapping_rotation_analysis.raw_to_norm`) it must differ **only**
on legs shorter than 90 samples.

**Correction to the plan.** The plan expected the rebuild to *differ* from `norm_neurons_dic` on
`ah10_20250618_20250619` session 5, on the grounds that the pickle predates the 6 Sep salvage. It
does not: comparing the current `data_dic` with `data_dic_lec.pkl.PRE_salvage_…` over all 25
recdays and every session, the salvage added `Locs_raw`, `XY_raw` and `HD_raw` to that one session
and changed **nothing else** — `Neuron_raw` and `Trial_times`, which are all these curves depend on,
are identical everywhere. What the salvage does change for us is the **session set**:
`get_sessions_for_glm` requires `Locs_raw`, so session 5 was invisible before it and is a fold now.
That is the currency test below.

In [3]:
# PFC has no legacy normalised arrays to check against (`build_data_dic_from_pfc` builds
# `Neurons_norm` only on request and this analysis never uses it). The equivalent guarantee here
# is control 1 of the synthetics (the rebinning is `scipy.stats.binned_statistic`) plus the fact
# that the curves come from the same files the GLM fits read.
print('sessions per recday used (get_sessions_for_glm):')
display(pd.DataFrame([{'recday': rd, 'sessions': c['sessions'], 'tasks': len(set(c['tasks'].values()))}
                      for rd, c in caches.items()]))

sessions per recday used (get_sessions_for_glm):


,recday,sessions,tasks
0,ab03_01092023_02092023,"[0, 1, 2, 3, 4, 5]",6
1,ab03_05092023_06092023,"[0, 1, 2, 4, 5, 6]",6
2,ab03_29082023_30082023,"[0, 1, 2, 4, 6, 7]",6
3,ah03_12082021_13082021,"[1, 2, 3, 4, 5, 6]",6
4,ah03_18082021_19082021,"[1, 2, 3, 5, 6, 7]",6
5,ah04_01122021_02122021,"[1, 2, 3, 5, 6, 7]",6
6,ah04_05122021_06122021,"[0, 1, 2, 4, 5, 6]",6
7,ah04_07122021_08122021,"[0, 1, 3, 5, 6, 7]",6
8,ah04_09122021_10122021,"[1, 2, 3, 5, 6, 7]",6
9,ah04_14122021_16122021,"[1, 2, 3, 4, 5, 6]",6


## 2. Part A — sorted β heatmaps per arm

Rows are neurons, columns the ten goal-progress bins with bin 0 (the reference, exactly 0 by
construction), each row divided by its own max|β| so colour is shape not rate, sorted by peak bin.

**No smoothing anywhere in this section.** These are the fitted β values, ten per neuron, plotted
as they come out of the GLM (`interpolation='none'`, no filter). The σ = 3 bin smoothing named in
the plan applies only to the 90-bin per-leg curves of Part B, and section 2.3 shows those at full
resolution with and without it.

**The sort manufactures the diagonal.** Synthetic control 7 plants a bump population and a pure
noise population and shows both give a clean diagonal; only the bumps have structure around the
peak (neighbour excess 0.63 against 0.01). Read the width of the bright band, not the band.

In [4]:
arms = {}
for section in G.ARMS:
    arms[section] = G.load_arm(section)
print(f'{len(arms)} arms loaded; recdays per arm:',
      {s.split("__")[1]: len(r["cv_results"]) for s, r in arms.items()})
region_of, GROUPS, COLORS = None, None, None

# The per-neuron table and the curve matrices are built here, before the figures, so that the
# A-B gate (when active) applies to EVERY panel downstream and not just the statistics.
T_all = G.assemble(caches, arms[G.SELECTING_ARM], arms[G.GP_ONLY_ARM],
                   )
M_all = G.stack_curves(caches, sigma=G.SIGMA)
assert (M_all['recday'] == T_all['recday'].to_numpy()).all() and \
       (M_all['neuron'].astype(int) == T_all['neuron'].to_numpy()).all(), 'curve/table row mismatch'
keep = (np.ones(len(T_all), bool) if AB_GATE is None
        else (T_all.ab_offset_bins <= AB_GATE).to_numpy())
T = T_all[keep].reset_index(drop=True)
M = {k: (v[keep] if isinstance(v, np.ndarray) and len(v) == len(keep) else v)
     for k, v in M_all.items()}
S = T[T.sel_sig_gp]
SIG_KEYS = set(zip(S['recday'], S['neuron'].astype(int)))
# GATE_KEYS narrows the beta heatmaps to the same population. Over ALL gated neurons, not just the
# gp-significant ones, because the `_all` heatmaps include non-significant cells. None in the
# primary run, so those heatmaps keep reproducing the plot_beta_profile selection exactly.
GATE_KEYS = (None if AB_GATE is None
             else set(zip(T['recday'], T['neuron'].astype(int))))
print(f'{len(T_all)} neurons; kept by the gate {len(T)}; gp-significant {len(S)}')

6 arms loaded; recdays per arm: {'matched_250ms_decile_cap30s_tfrD10b': 25, 'matched_250ms_decile_cap30s_tfrU10b': 25, 'matched_250ms_decile_cap60s_tfrD10b': 25, 'matched_250ms_decile_cap60s_tfrU10b': 25, 'matched_250ms_decile_cap30s': 25, 'matched_250ms_decile_cap60s': 25}


1252 neurons; kept by the gate 1173; gp-significant 930


In [5]:
counts = []
for section, res in arms.items():
    tag = section.split('__')[1]
    for reg in (['goal_progress', 'time_from_reward'] if 'progress_time' in section else ['goal_progress']):
        col_idx = glm._resolve_regressor_groups(G.regs_for(section), parameterization='reference_coded')[0][reg]
        short = 'gp' if reg == 'goal_progress' else 'tfr'
        for only_sig, suffix in ((True, ''), (False, '_all')):
            fig, info = G.plot_beta_heatmap(res['glm_results'], res['cv_results'], col_idx, reg,
                                            only_significant=only_sig, keep_keys=GATE_KEYS,
                                            out_path=f'{FIGDIR}/pfc_{short}_beta_heatmap_{tag}{suffix}.pdf')
            plt.close(fig)
            counts.append({'arm': tag, 'regressor': reg, 'significant only': only_sig,
                           'rows': info['n_rows'], 'dropped by gate': info['n_dropped_by_keep_keys']})
            if region_of is not None:
                figr, infor = G.plot_beta_heatmap(res['glm_results'], res['cv_results'], col_idx, reg,
                                                  only_significant=only_sig, region_of=region_of,
                                                  groups=GROUPS, colors=COLORS, keep_keys=GATE_KEYS,
                                                  out_path=f'{FIGDIR}/pfc_{short}_beta_heatmap_{tag}{suffix}_regions.pdf')
                counts[-1]['rows (regions)'] = infor['n_rows']
                counts[-1]['per region'] = infor['n_per_block']
                plt.close(figr)
display(pd.DataFrame(counts))

,arm,regressor,significant only,rows,dropped by gate
0,matched_250ms_decile_cap30s_tfrD10b,goal_progress,True,804,32
1,matched_250ms_decile_cap30s_tfrD10b,goal_progress,False,1173,79
2,matched_250ms_decile_cap30s_tfrD10b,time_from_reward,True,854,48
3,matched_250ms_decile_cap30s_tfrD10b,time_from_reward,False,1173,79
4,matched_250ms_decile_cap30s_tfrU10b,goal_progress,True,930,44
5,matched_250ms_decile_cap30s_tfrU10b,goal_progress,False,1173,79
6,matched_250ms_decile_cap30s_tfrU10b,time_from_reward,True,661,38
7,matched_250ms_decile_cap30s_tfrU10b,time_from_reward,False,1173,79
8,matched_250ms_decile_cap60s_tfrD10b,goal_progress,True,800,33
9,matched_250ms_decile_cap60s_tfrD10b,goal_progress,False,1173,79


### 2.1 Row counts against the β-profile legend

`plot_beta_heatmap` must select exactly the neurons `glm_plots.plot_beta_profile` averages, or the
two figures of the same arm are about different populations.

In [6]:
section = G.SELECTING_ARM
res = arms[section]
col_idx = G.gp_col_idx(section)
figp = P.plot_beta_profile(res['glm_results'], res['cv_results'], col_idx, 'goal_progress',
                           region_of=region_of, groups=GROUPS, colors=COLORS)
legend = G.legend_n(figp)
figh, info = G.plot_beta_heatmap(res['glm_results'], res['cv_results'], col_idx, 'goal_progress',
                                 region_of=region_of, groups=GROUPS, colors=COLORS,
                                 keep_keys=GATE_KEYS)
print('profile legend (never gated -- glm_plots is not modified):', legend)
print('heatmap blocks :', info['n_per_block'])
if AB_GATE is None:
    assert legend == info['n_per_block'], 'heatmap rows disagree with the beta-profile n'
    print('exact match: the heatmap shows the population the profile averages')
else:
    # the profile cannot be gated (its module is untouchable), so the check becomes: every block
    # is a SUBSET of the ungated one, and the total drop is what the gate removed
    assert all(info['n_per_block'][g] <= legend[g] for g in info['n_per_block']), \
        'a gated heatmap block is larger than the ungated beta-profile n'
    drop = {g: legend[g] - info['n_per_block'][g] for g in info['n_per_block']}
    # the total the gate removed also covers regions the profile does not draw (fibre/other),
    # so the drawn regions can only account for part of it
    assert 0 <= sum(drop.values()) <= info['n_dropped_by_keep_keys'], \
        'the per-region drop is inconsistent with the number of rows the gate removed'
    print('dropped by the gate, per region:', drop,
          f"({sum(drop.values())} of {info['n_dropped_by_keep_keys']} across all regions)")
plt.show()

profile legend (never gated -- glm_plots is not modified): {'all': 974}
heatmap blocks : {'all': 930}
dropped by the gate, per region: {'all': 44} (44 of 44 across all regions)


### 2.2 The selecting arm, on screen

The uniform/30 arm is where the goal-progress cells are defined. Significant-only and all-neuron,
pooled.

In [7]:
for only_sig in (True, False):
    fig, info = G.plot_beta_heatmap(res['glm_results'], res['cv_results'], col_idx, 'goal_progress',
                                    only_significant=only_sig)
    fig.suptitle(f"goal progress, uniform/30 — {'gp-significant' if only_sig else 'all'} "
                 f"neurons (n={info['n_rows']})", fontsize=7)
    plt.show()

### 2.3 The same population at full resolution — 90 bins, unsmoothed and smoothed

The β heatmaps above have ten columns because the GLM has ten goal-progress columns. The per-leg
curves have ninety, and no smoothing is required to draw them. Three versions, so the smoothing is
visible rather than assumed:

* **σ = 0**, sorted on itself — raw bin means, the classic picture, and the diagonal is circular;
* **σ = 3 bins**, sorted on itself — what the width estimator sees;
* **cross-validated** — rows sorted by the ODD-leg peak, colours from the EVEN legs. Here a
  diagonal is evidence: the sort knows nothing about the legs being shown. Compare its sharpness
  with the two above to see how much of their diagonal is the sort.

In [8]:
# the same cells the statistics use (gp-significant, and inside the gate when one is active)
for sg in (0, G.SIGMA):
    Ms = G.stack_curves(caches, sigma=sg)
    key = list(zip(Ms['recday'], Ms['neuron'].astype(int)))
    sel = np.array([k in SIG_KEYS for k in key])
    lab = (np.array([region_of[rd][n] for rd, n in np.asarray(key, dtype=object)[sel]])
           if region_of is not None else None)
    fig, info = G.plot_curve_heatmap(Ms['full'][sel], Ms['full'][sel],
                                     title=f'all legs, sigma={sg} bins, sorted on itself (circular) — n={sel.sum()}',
                                     out_path=f'{FIGDIR}/pfc_curve_heatmap_sigma{sg}.pdf')
    plt.show()
    fig, info = G.plot_curve_heatmap(Ms['A'][sel], Ms['B'][sel],
                                     title=f'cross-validated: sorted by odd legs, shown from even, sigma={sg}',
                                     out_path=f'{FIGDIR}/pfc_curve_heatmap_xval_sigma{sg}.pdf')
    plt.show()
    

## 3. Part B — the estimator, calibrated

Planted phase cells (width 0.05–0.3 at peaks 0.1–0.9) and time cells (latency and lead 0.5–12 s) on
each recday's real legs, through `build_curves` → `split_half` → `peak_and_width` unchanged.

In [9]:
fig = G.plot_calibration(SIMS[SIMS['sigma'] == G.SIGMA],
                        out_path=f'{FIGDIR}/pfc_width_calibration.pdf'); plt.show()
fig = G.plot_time_cell_null(NULL, SIMS,
                            out_path=f'{FIGDIR}/pfc_time_cell_null.pdf'); plt.show()
display(NULL[NULL.kind.isin(['tfr', 'ttr'])][['kind', 'param', 'peak_med', 'width_med',
                                              'frac_not_reproduced']].round(3))
print('smoothing sensitivity — recovered width of a planted 0.1 field:',
      SYN['results']['3  width / peak calibration (phase cells)']['info']['width_0.1_recovered_by_sigma'])

,kind,param,peak_med,width_med,frac_not_reproduced
55,tfr,0.5,0.128,0.200,0.000
56,tfr,1.0,0.194,0.256,0.000
57,tfr,2.0,0.317,0.378,0.040
58,tfr,3.0,0.461,0.489,0.000
59,tfr,4.0,0.617,0.522,0.000
60,tfr,6.0,0.794,0.489,0.013
61,tfr,8.0,0.872,0.411,0.013
62,tfr,10.0,0.922,0.333,0.013
63,tfr,12.0,0.906,0.300,0.053
64,ttr,0.5,0.872,0.200,0.000


smoothing sensitivity — recovered width of a planted 0.1 field: {2: 0.1, 3: 0.111, 5: 0.156}


## 4. The cells, joined

Curve route (peak, width, reliability, cross-session stability) joined to the β route of the
selecting arm (`sel_`) and the gp-only/30 arm (`gpo_`). The join is
positional and refuses a neuron-count mismatch.

In [10]:
print(f'{len(T_all)} neurons; gp-significant in the selecting arm: {T_all.sel_sig_gp.mean():.1%}; '
      f'in gp-only/30: {T_all.gpo_sig_gp.mean():.1%}')
if AB_GATE is not None:
    g0 = T_all[T_all.sel_sig_gp].dropna(subset=['width_hm'])
    g1 = T[T.sel_sig_gp].dropna(subset=['width_hm'])
    print(f'A-B gate at {AB_GATE} bins: {len(g1)} of {len(g0)} gp-significant cells with a defined '
          f'width kept ({len(g1)/len(g0):.1%}); median width {g0.width_hm.median():.3f} -> '
          f'{g1.width_hm.median():.3f}')
flags = pd.DataFrame({'all cells': T[['peak_not_reproduced', 'multimodal', 'ramp', 'flat', 'remaps']].mean(),
                      'gp-significant': S[['peak_not_reproduced', 'multimodal', 'ramp', 'flat', 'remaps']].mean()})
display(flags.round(3))
print(f"width defined for {np.isfinite(S.width_hm).mean():.1%} of gp-significant cells; "
      f"median split-half r {S.splithalf_r.median():.3f}")
display(S.groupby('group')[['splithalf_r', 'width_hm', 'peak_phase', 'max_pair_dist_bins']]
        .agg(['count', 'median']).round(3))

1252 neurons; gp-significant in the selecting arm: 77.8%; in gp-only/30: 86.0%
A-B gate at 30 bins: 898 of 934 gp-significant cells with a defined width kept (96.1%); median width 0.489 -> 0.489


,all cells,gp-significant
peak_not_reproduced,0.057,0.034
multimodal,0.234,0.185
ramp,0.037,0.035
flat,0.000,0.000
remaps,0.482,0.409


width defined for 96.6% of gp-significant cells; median split-half r 0.928


splithalf_r        width_hm        peak_phase        max_pair_dist_bins  \
            count median    count median      count median              count   
group                                                                           
PFC           930  0.928      898  0.489        930   0.55                930   

              
      median  
group         
PFC     27.0

### 4.0 How the width is measured, on real cells

The rule in one line: **width = the length of the contiguous circular run of the even-leg curve B
above B's OWN half-max, that contains the odd-leg peak bin.**

The threshold is half way between B's minimum and B's maximum — **not** half of B's value at the
odd-leg peak. If B sits at 0.8 of its own range at that bin, the cutoff is still 0.5 of the range,
not 0.4. Two consequences, both deliberate:

* the run is generally **not symmetric** about the odd-leg peak. It is B's own field; the odd-leg
  peak only selects *which* field to measure, so that noise in B's argmax cannot make the estimator
  jump to a different bump.
* if B is below its own half-max at that bin the peak did not reproduce: no run, width NaN, cell
  flagged and counted.

One property to keep in mind when reading the panels: the curve is min-max normalised for the
threshold, so the width is a measure of **shape, not modulation depth**. A 1 Hz ripple on a 12 Hz
baseline and a 15 Hz field on a 1 Hz baseline are scored the same way. Depth lives in `range_hz_B`
and reliability in `splithalf_r`, both carried per neuron.

In [11]:
sel = T.sel_sig_gp.to_numpy()
fig, picks = G.plot_width_explainer(M['A'][sel], M['B'][sel], T[sel],
                                    out_path=f'{FIGDIR}/pfc_width_explainer.pdf')
plt.show()
display(T[sel].iloc[picks][['recday', 'neuron', 'group', 'peak_phase', 'width_hm', 'width_hm_linear',
                            'width_csd', 'above_half_frac', 'multimodal', 'peak_not_reproduced',
                            'splithalf_r', 'range_hz_B']].round(3))

,recday,neuron,group,peak_phase,width_hm,width_hm_linear,width_csd,above_half_frac,multimodal,peak_not_reproduced,splithalf_r,range_hz_B
1096,me11_05122021_06122021,30,PFC,0.983,0.133,0.111,0.137,0.133,False,False,0.821,4.289
689,ah07_01092023_02092023,25,PFC,0.461,0.756,0.756,0.256,0.756,False,False,0.979,9.352
664,ah04_14122021_16122021,68,PFC,0.961,0.222,0.200,0.177,0.222,False,False,0.978,1.845
308,ah04_01122021_02122021,96,PFC,0.494,0.556,0.556,0.270,0.689,True,False,0.872,1.216
157,ab03_29082023_30082023,37,PFC,0.361,0.700,0.700,0.257,0.700,False,False,0.966,2.757
35,ab03_01092023_02092023,35,PFC,0.594,NaN,NaN,0.228,0.356,False,True,0.409,8.835


### 4.0b What A's peak is for, and a stricter gate

A's peak has two jobs: it **is** the reported peak phase (and it must come from held-out legs, or
peak and width would be coupled by the same noise), and it selects which run of B to measure. The
consistency check is a by-product: a cell whose B curve is below its own half-max at A's peak is
dropped.

The distance between the two halves' peaks is described here in both runs. Whether to *gate* on it
is a separate question, and the sweep that answers it lives in the gated variant notebook
(`*_gp_tuning_width_abgate.ipynb`), not here — the primary run does not gate.

In [12]:
d = S.dropna(subset=['width_hm'])
print('A-vs-B peak offset (bins of 90):',
      {k: round(v, 1) for k, v in d.ab_offset_bins.describe(percentiles=[.5, .9]).to_dict().items()})
nr = S[S.peak_not_reproduced]
print(f"dropped by the current rule (B below its own half-max at A's peak): {len(nr)} "
      f"({len(nr)/len(S):.1%}), median split-half r {nr.splithalf_r.median():.3f} "
      f"vs {d.splithalf_r.median():.3f} for the kept cells")

A-vs-B peak offset (bins of 90): {'count': 898.0, 'mean': 6.5, 'std': 6.9, 'min': 0.0, '50%': 4.0, '90%': 17.0, 'max': 30.0}
dropped by the current rule (B below its own half-max at A's peak): 32 (3.4%), median split-half r 0.437 vs 0.933 for the kept cells


#### 4.0c The gate sweep — variant notebook only

How both correlations move as the gate tightens, with the planted populations gated **identically**
at every threshold. Computed on the ungated table, so the curve starts at "none" and this run sits
at one point on it. This figure is written only to the variant directory: a sweep over whether to
gate is not part of the primary analysis.

In [13]:
sweep = G.ab_gate_sweep(T_all[T_all.sel_sig_gp].assign(
                            abs_peak_off_centre=lambda x: np.abs(x.peak_phase - .5)),
                        SYN['sims'][SYN['sims'].sigma == G.SIGMA],
                        groups=['PFC'])
display(sweep.round(3))
fig = G.plot_ab_gate_sweep(sweep, out_path=f'{FIGDIR}/pfc_ab_gate_sweep.pdf'); plt.show()
print('Read the sweep, not one cut: the gate removes wide cells (they are the disagreeing ones), '
      'and wide cells are disproportionately mid-leg, so it selects on something related to the '
      'outcome. The ungated run stays primary.')

,threshold_bins,population,n,kept,median_width,rho_peak_width,rho_absoff_width,V PFC,mice PFC
0,none,real,934,1.000,0.489,0.234,-0.554,-0.521,7.0
1,none,sim time cells,1330,1.000,0.378,-0.001,-0.482,NaN,NaN
2,none,sim phase cells,4050,1.000,0.250,-0.012,-0.005,NaN,NaN
3,45,real,934,1.000,0.489,0.234,-0.554,-0.521,7.0
4,45,sim time cells,1330,1.000,0.378,-0.001,-0.482,NaN,NaN
5,45,sim phase cells,4050,1.000,0.250,-0.012,-0.005,NaN,NaN
6,30,real,898,0.961,0.489,0.232,-0.576,-0.573,7.0
7,30,sim time cells,1323,0.995,0.378,-0.003,-0.482,NaN,NaN
8,30,sim phase cells,4050,1.000,0.250,-0.012,-0.005,NaN,NaN
9,20,real,843,0.903,0.478,0.238,-0.587,-0.594,7.0


Read the sweep, not one cut: the gate removes wide cells (they are the disagreeing ones), and wide cells are disproportionately mid-leg, so it selects on something related to the outcome. The ungated run stays primary.


### 4.1 Cross-session peak stability

Per-session peaks on the circular 90-bin leg axis; a cell is flagged if the **maximum pairwise
circular distance** between its per-session peaks exceeds **30 bins** (a third of the leg) — the
statistic chosen with the user in place of the plan's circular SD, which saturates (two peaks 30
bins apart give an SD of only ~17 bins). Both are reported.

In [14]:
st = S.dropna(subset=['max_pair_dist_bins'])
display(st.groupby('group')[['max_pair_dist_bins', 'circ_sd_bins']].agg(['count', 'median']).round(2))
print(f"remapping fraction (max pairwise > 30 bins): {st.remaps.mean():.1%}")
display(st.groupby('group')['remaps'].agg(['mean', 'size']).round(3))
fig, a = plt.subplots(1, 2, figsize=(5.2, 2.2))
a[0].hist(st.max_pair_dist_bins, bins=np.arange(0, 46, 2), color=G.C_NEUTRAL, edgecolor='none')
a[0].axvline(G.REMAP_BINS, color=G.C_TIME, lw=1, ls='--'); a[0].set_xlabel('max pairwise peak distance (bins)')
a[0].set_ylabel('cells')
a[1].scatter(st.max_pair_dist_bins, st.circ_sd_bins, s=4, alpha=.4, color=G.C_NEUTRAL, edgecolor='none', rasterized=True)
a[1].set_xlabel('max pairwise (bins)'); a[1].set_ylabel('circular SD (bins)')
fig.tight_layout(); G._save(fig, f'{FIGDIR}/pfc_peak_stability.pdf'); plt.show()

max_pair_dist_bins        circ_sd_bins       
                   count median        count median
group                                              
PFC                  930   27.0          930   9.44

remapping fraction (max pairwise > 30 bins): 40.9%


,mean,size
group,,
PFC,0.409,930


## 5. Peak vs width, against the simulated V

Colour is the gp-vs-tfr split in the selecting arm (Δr²_gp − Δr²_tfr): blue leans absolute time,
red leans phase.

In [15]:
fig = G.plot_peak_vs_width(S, NULL, title=f'PFC, gp-significant (uniform/30), n={len(S)}',
                           out_path=f'{FIGDIR}/pfc_peak_vs_width.pdf'); plt.show()
# the same data with no simulation drawn over it
fig = G.plot_peak_vs_width(S, NULL, overlay=False,
                           title=f'PFC, gp-significant (uniform/30), n={len(S)} — data only',
                           out_path=f'{FIGDIR}/pfc_peak_vs_width_nooverlay.pdf'); plt.show()
G2 = T[T.gpo_sig_gp]
fig = G.plot_peak_vs_width(G2, NULL, title=f'gp-only/30 significant (any within-leg structure), n={len(G2)}',
                           out_path=f'{FIGDIR}/pfc_peak_vs_width_gponly.pdf'); plt.show()
fig = G.plot_peak_vs_width(G2, NULL, overlay=False,
                           title=f'gp-only/30 significant, n={len(G2)} — data only',
                           out_path=f'{FIGDIR}/pfc_peak_vs_width_gponly_nooverlay.pdf'); plt.show()

### 5.1 The statistics — neuron within mouse

ρ(peak, width) and ρ(|peak − 0.5|, width), per (recday, region) → mouse mean → mean over mice, with
the mice shown. The dashed lines are the same statistics computed on the simulated populations:
time cells give a strongly negative ρ(|peak − 0.5|, width) (the V), phase cells give zero.

In [16]:
sim = SIMS[(SIMS['sigma'] == G.SIGMA)].dropna(subset=['width_hm'])
from scipy.stats import spearmanr
sim_ref = {}
for lab, sub in (('simulated time cells', sim[sim.kind.isin(['tfr', 'ttr'])]),
                 ('simulated phase cells', sim[sim.kind == 'phase'])):
    sim_ref[lab] = {'rho_peak_width': float(spearmanr(sub.peak_phase, sub.width_hm).correlation),
                    'rho_absoff_width': float(spearmanr(np.abs(sub.peak_phase - .5), sub.width_hm).correlation)}
display(pd.DataFrame(sim_ref).T.round(3))

for x, lab, key in (('peak_phase', r'$\rho$(peak, width)', 'rho_peak_width'),
                    ('abs_peak_off_centre', r'$\rho$(|peak - 0.5|, width)', 'rho_absoff_width')):
    rep = G.rho_report(S, x, 'width_hm', groups=['PFC'])
    display(rep.round(3))
    fig = G.plot_rho_by_group(rep, colors=COLORS, ylabel=lab,
                              sim_rows={k: v[key] for k, v in sim_ref.items()},
                              title=f'gp-significant cells, uniform/30',
                              out_path=f'{FIGDIR}/pfc_rho_{x}.pdf')
    plt.show()

,rho_peak_width,rho_absoff_width
simulated time cells,-0.003,-0.482
simulated phase cells,-0.012,-0.005


,group,n_mice,n_recdays,n_units,rho_mean_over_mice,ci_lo,ci_hi,per_mouse,evidence
0,PFC,7,22,885,0.244,0.084,0.44,"{'ab03': 0.267, 'ah03': 0.15, 'ah04': 0.136, '...",primary-capable


/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/code/gp_tuning_width.py:1734: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


,group,n_mice,n_recdays,n_units,rho_mean_over_mice,ci_lo,ci_hi,per_mouse,evidence
0,PFC,7,22,885,-0.573,-0.686,-0.465,"{'ab03': -0.488, 'ah03': -0.509, 'ah04': -0.52...",primary-capable


/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/code/gp_tuning_width.py:1734: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


### 5.1a How many cells peak in a bin, against how wide they are there

Bars are the number of gp-significant cells whose peak falls in each tenth of the leg; the line is
the median half-max width of exactly those cells. The right panel puts the two against each other,
one point per bin, with the simulated populations as crosses. The correlation there is across ten
bins, not across cells.

In [17]:
fig, cw = G.plot_peak_count_vs_width(S, sim_table=sim,
                                     title=f'PFC, gp-significant (uniform/30)',
                                     out_path=f'{FIGDIR}/pfc_peak_count_vs_width.pdf')
plt.show()
# the same figure with no simulated populations drawn in the right panel
fig, _ = G.plot_peak_count_vs_width(S, sim_table=None,
                                    title=f'PFC, gp-significant (uniform/30) — data only',
                                    out_path=f'{FIGDIR}/pfc_peak_count_vs_width_nosim.pdf')
plt.show()
display(cw.round(3))
print(f"across bins: rho = {cw.attrs['spearman_count_vs_width']:+.3f}, p = {cw.attrs['p']:.3g}, "
      f"{cw.attrs['n_cells']} cells in 10 bins")

/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/code/gp_tuning_width.py:1460: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, (a, b) = plt.subplots(1, 2, figsize=(6.0, 2.6))
/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/code/gp_tuning_width.py:1512: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  b.legend(frameon=False, fontsize=5, loc='lower right')


,bin,phase,n_peaking,frac_peaking,width_median,width_lo,width_hi,width_mean
0,0,0.05,135,0.150,0.144,0.122,0.256,0.202
1,1,0.15,67,0.075,0.389,0.333,0.450,0.394
2,2,0.25,63,0.070,0.478,0.406,0.567,0.484
3,3,0.35,70,0.078,0.511,0.411,0.653,0.521
4,4,0.45,82,0.091,0.600,0.492,0.719,0.610
5,5,0.55,76,0.085,0.617,0.464,0.722,0.588
6,6,0.65,82,0.091,0.583,0.511,0.689,0.582
7,7,0.75,97,0.108,0.633,0.522,0.711,0.607
8,8,0.85,75,0.084,0.556,0.422,0.622,0.517
9,9,0.95,151,0.168,0.300,0.144,0.567,0.360


across bins: rho = -0.024, p = 0.947, 898 cells in 10 bins


### 5.1b The two ρ's, and what each simulated population predicts

The two statistics say different things and the simulation gives a reference for each **per anchor**,
which the pooled "time cells" line hides. A purely retrospective population (fields at a fixed
latency after reward) has its width rise with peak phase over most of the leg, so its ρ(peak, width)
is **positive**; a purely prospective one mirrors it and is **negative**; an equal mixture cancels to
about zero on ρ(peak, width) while both give the same strongly negative ρ(|peak − 0.5|, width).
So ρ(|peak − 0.5|, width) is the V, and ρ(peak, width) is the *anchor asymmetry*.

In [18]:
rows = []
for lab, sub in (('sim time cells, both anchors', sim[sim.kind.isin(['tfr', 'ttr'])]),
                 ('sim retrospective only (tfr)', sim[sim.kind == 'tfr']),
                 ('sim prospective only (ttr)', sim[sim.kind == 'ttr']),
                 ('sim phase cells', sim[sim.kind == 'phase']),
                 ('sim noise cells', sim[sim.kind == 'noise']),
                 ('REAL gp-significant (pooled)', S.dropna(subset=['width_hm']))):
    rows.append({'population': lab, 'n': len(sub),
                 'rho(peak, width)': round(float(spearmanr(sub.peak_phase, sub.width_hm).correlation), 3),
                 'rho(|peak-0.5|, width)': round(float(spearmanr(np.abs(sub.peak_phase - .5), sub.width_hm).correlation), 3),
                 'median width': round(float(sub.width_hm.median()), 3),
                 'median |peak-0.5|': round(float(np.abs(sub.peak_phase - .5).median()), 3)})
display(pd.DataFrame(rows))

,population,n,"rho(peak, width)","rho(|peak-0.5|, width)",median width,median |peak-0.5|
0,"sim time cells, both anchors",1323,-0.003,-0.482,0.378,0.317
1,sim retrospective only (tfr),661,0.289,-0.487,0.378,0.306
2,sim prospective only (ttr),662,-0.291,-0.477,0.378,0.317
3,sim phase cells,4050,-0.012,-0.005,0.250,0.206
4,sim noise cells,530,0.058,-0.059,0.200,0.239
5,REAL gp-significant (pooled),898,0.232,-0.576,0.489,0.283


### 5.1c Restricted to phase-stable cells

Half the gp-significant cells move their peak by more than a third of a leg between sessions
(section 4.1). Pooling their legs across sessions must widen their curve, and a cell whose peak
lands mid-leg after such averaging would be counted broad for a reason that has nothing to do with
time coding. The relation is therefore recomputed on the stable cells alone, and within the single
session with the most legs.

In [19]:
stable = S[(~S.remaps.fillna(True))].dropna(subset=['width_hm'])
print(f'{len(stable)} of {len(S.dropna(subset=["width_hm"]))} gp-significant cells are phase-stable '
      f'(max pairwise peak distance <= {G.REMAP_BINS} bins)')
rows = []
for x, key in (('peak_phase', 'rho(peak, width)'), ('abs_peak_off_centre', 'rho(|peak-0.5|, width)')):
    for lab, sub in (('all gp-significant', S), ('phase-stable only', stable)):
        rep = G.rho_report(sub, x, 'width_hm', groups=['PFC'])
        for _, r in rep.iterrows():
            rows.append({'statistic': key, 'set': lab, 'group': r['group'], 'n_mice': r['n_mice'],
                         'n_units': r['n_units'], 'rho': round(r['rho_mean_over_mice'], 3),
                         'ci': (round(r['ci_lo'], 3), round(r['ci_hi'], 3))})
display(pd.DataFrame(rows).pivot_table(index=['statistic', 'group'], columns='set',
                                       values='rho', aggfunc='first'))

# and within one session only -- no cross-session averaging at all
rows = []
for rd, c in caches.items():
    ls = c['leg_session']
    best = max(set(ls.tolist()), key=lambda s: (ls == s).sum())
    m = ls == best
    if m.sum() < 40:
        continue
    A, B, r, _ = G.split_half(c['curves'][:, m, :], sigma=G.SIGMA)
    t = G.peak_and_width(A, B)
    t['recday'] = rd; t['mouse'] = rd.split('_')[0]; t['neuron'] = np.arange(len(t))
    t['splithalf_r'] = r
    rows.append(t)
W1 = pd.concat(rows, ignore_index=True).merge(T[['recday', 'neuron', 'sel_sig_gp', 'group']],
                                              on=['recday', 'neuron'])
W1 = W1[W1.sel_sig_gp].assign(abs_peak_off_centre=lambda d: np.abs(d.peak_phase - .5))
print(f'single richest session per recday: {len(W1)} gp-significant cells, '
      f'width defined for {np.isfinite(W1.width_hm).mean():.1%}, median width {W1.width_hm.median():.3f}')
for x, key in (('peak_phase', 'rho(peak, width)'), ('abs_peak_off_centre', 'rho(|peak-0.5|, width)')):
    rep = G.rho_report(W1, x, 'width_hm', groups=['PFC'])
    print(key, {r['group']: round(r['rho_mean_over_mice'], 3) for _, r in rep.iterrows()})

541 of 898 gp-significant cells are phase-stable (max pairwise peak distance <= 30 bins)


,set,all gp-significant,phase-stable only
statistic,group,,
"rho(peak, width)",PFC,0.244,0.309
"rho(|peak-0.5|, width)",PFC,-0.573,-0.554


single richest session per recday: 930 gp-significant cells, width defined for 77.5%, median width 0.378


rho(peak, width) {'PFC': 0.154}


rho(|peak-0.5|, width) {'PFC': -0.473}


### 5.2 Stratified by the gp-vs-tfr split

If the relation is time coding it should weaken or vanish in the cells that lean phase.

In [20]:
med = S.sel_split.median()
strata = {'leans time (split < median)': S[S.sel_split < med], 'leans phase (split >= median)': S[S.sel_split >= med]}
rows = []
for lab, sub in strata.items():
    for x, key in (('peak_phase', 'rho(peak, width)'), ('abs_peak_off_centre', 'rho(|peak-0.5|, width)')):
        rep = G.rho_report(sub, x, 'width_hm', groups=['PFC'])
        for _, r in rep.iterrows():
            rows.append({'stratum': lab, 'statistic': key, 'group': r['group'], 'n_mice': r['n_mice'],
                         'rho': round(r['rho_mean_over_mice'], 3), 'ci_lo': round(r['ci_lo'], 3),
                         'ci_hi': round(r['ci_hi'], 3), 'n_units': r['n_units']})
display(pd.DataFrame(rows))

,stratum,statistic,group,n_mice,rho,ci_lo,ci_hi,n_units
0,leans time (split < median),"rho(peak, width)",PFC,5,0.334,0.080,0.626,410
1,leans time (split < median),"rho(|peak-0.5|, width)",PFC,5,-0.404,-0.544,-0.272,410
2,leans phase (split >= median),"rho(peak, width)",PFC,6,0.205,0.009,0.364,439
3,leans phase (split >= median),"rho(|peak-0.5|, width)",PFC,6,-0.602,-0.717,-0.474,439


### 5.2b Edge artefacts in the half-max width

The width is a contiguous run **on the circle**, so a field sitting on the leg boundary is never
truncated and no peak position has less room than any other. Three measurements of that claim,
because "no edge by construction" is an argument and not evidence:

1. **Calibration in the regime the data occupy.** Control 3 plants fields of width 0.05–0.5 at
   peaks 0.1–0.9 and reports the recovered width per planted peak. A peak-dependence of zero bins
   is the absence of an edge effect. The plan only asked for 0.05–0.3; 0.4 and 0.5 were added
   because the real median width is about 0.49 and a calibration that stops short of the data says
   nothing about it.
2. **What a non-circular definition would have cost.** `width_hm_linear` is the same run with the
   wrap forbidden. Its deficit against the circular width, per peak decile, is the artefact the
   circular axis avoids — and it lands, as it must, on the cells whose field straddles the reward.
3. **The threshold-free width.** `width_csd` (circular SD of the baseline-subtracted curve as a
   density) uses no half-max threshold at all, so it cannot have a threshold-crossing edge. If the
   relation survives with it, the half-max rule is not producing it.

In [21]:
cal = SYN['results']['3  width / peak calibration (phase cells)']['info']
display(pd.DataFrame([{'planted width': k.replace('planted_width_', ''),
                       'peak-dependence (bins)': v['peak_dependence_bins'],
                       'max abs error (bins)': v['max_abs_err_bins'],
                       'gated': v.get('pass', 'recorded only')}
                      for k, v in cal.items() if k.startswith('planted_width_')]))

d = S.dropna(subset=['width_hm'])
dec = pd.cut(d.peak_phase, np.linspace(0, 1, 11), include_lowest=True)
edge = d.groupby(dec, observed=True).agg(
    n=('width_hm', 'size'), circular=('width_hm', 'median'), linear=('width_hm_linear', 'median'),
    circ_sd=('width_csd', 'median'), above_half=('above_half_frac', 'median'),
    multimodal=('multimodal', 'mean'))
edge['linear deficit'] = edge['circular'] - edge['linear']
display(edge.round(3))
print(f"cells whose half-max run wraps the reward (circular > linear): "
      f"{(d.width_hm > d.width_hm_linear + 1e-9).mean():.1%}")

for x, key in (('peak_phase', 'rho(peak, .)'), ('abs_peak_off_centre', 'rho(|peak-0.5|, .)')):
    row = {'statistic': key}
    for wcol, lab in (('width_hm', 'half-max (circular)'), ('width_hm_linear', 'half-max (linear)'),
                      ('width_csd', 'circular SD (threshold-free)')):
        rep = G.rho_report(S, x, wcol, groups=['PFC'])
        row[lab] = {r['group']: round(r['rho_mean_over_mice'], 3) for _, r in rep.iterrows()}
    display(pd.DataFrame([row]).T)

,planted width,peak-dependence (bins),max abs error (bins),gated
0,0.05,0.0,3.5,recorded only
1,0.1,0.0,1.0,True
2,0.2,0.0,0.0,True
3,0.3,1.0,1.0,True
4,0.4,0.0,0.0,True
5,0.5,1.0,1.0,True


,n,circular,linear,circ_sd,above_half,multimodal,linear deficit
peak_phase,,,,,,,
"(-0.001, 0.1]",135,0.144,0.100,0.187,0.211,0.230,0.044
"(0.1, 0.2]",67,0.389,0.378,0.201,0.400,0.164,0.011
"(0.2, 0.3]",63,0.478,0.478,0.214,0.511,0.190,0.000
"(0.3, 0.4]",70,0.511,0.511,0.222,0.550,0.214,0.000
"(0.4, 0.5]",82,0.600,0.600,0.239,0.656,0.159,0.000
"(0.5, 0.6]",76,0.617,0.617,0.238,0.639,0.184,0.000
"(0.6, 0.7]",82,0.583,0.583,0.230,0.589,0.171,0.000
"(0.7, 0.8]",97,0.633,0.633,0.231,0.633,0.072,0.000
"(0.8, 0.9]",75,0.556,0.556,0.222,0.556,0.120,0.000


cells whose half-max run wraps the reward (circular > linear): 30.6%


,0
statistic,"rho(peak, .)"
half-max (circular),{'PFC': 0.244}
half-max (linear),{'PFC': 0.233}
circular SD (threshold-free),{'PFC': 0.21}


,0
statistic,"rho(|peak-0.5|, .)"
half-max (circular),{'PFC': -0.573}
half-max (linear),{'PFC': -0.625}
circular SD (threshold-free),{'PFC': -0.275}


### 5.2c Is the V just reliability? (the first objection)

A cell with no real tuning has a peak that lands anywhere, so unreliable cells are spread uniformly
over phase while reliable ones pile up where the fields actually are — at the leg edges here. If
unreliable cells are also scored broad, that alone would produce a V without any time coding. Three
checks: how width and reliability relate; where the unreliable cells peak; and the two ρ's
recomputed within the reliable half. The planted noise population is the reference — it went through
the same pipeline and gave ρ(|peak − 0.5|, width) ≈ +0.05, i.e. no V at all.

In [22]:
d = S.dropna(subset=['width_hm'])
print(f"rho(split-half r, width)      = {spearmanr(d.splithalf_r, d.width_hm).correlation:+.3f}")
print(f"rho(split-half r, |peak-0.5|) = {spearmanr(d.splithalf_r, d.abs_peak_off_centre).correlation:+.3f}")
q = pd.qcut(d.splithalf_r, 4, labels=['Q1 least reliable', 'Q2', 'Q3', 'Q4 most reliable'])
display(d.groupby(q, observed=True).agg(n=('width_hm', 'size'),
                                        splithalf_r=('splithalf_r', 'median'),
                                        width=('width_hm', 'median'),
                                        abs_off_centre=('abs_peak_off_centre', 'median'),
                                        frac_mid_leg=('abs_peak_off_centre', lambda v: (v < 0.2).mean())).round(3))
hi = d[d.splithalf_r >= d.splithalf_r.median()]
rows = []
for x, key in (('peak_phase', 'rho(peak, width)'), ('abs_peak_off_centre', 'rho(|peak-0.5|, width)')):
    for lab, sub in (('all gp-significant', d), ('reliable half (split-half r above median)', hi)):
        rep = G.rho_report(sub, x, 'width_hm', groups=['PFC'])
        for _, r in rep.iterrows():
            rows.append({'statistic': key, 'set': lab, 'group': r['group'],
                         'n_mice': r['n_mice'], 'n_units': r['n_units'],
                         'rho': round(r['rho_mean_over_mice'], 3)})
display(pd.DataFrame(rows).pivot_table(index=['statistic', 'group'], columns='set', values='rho',
                                       aggfunc='first'))
noise = sim[sim.kind == 'noise'].dropna(subset=['width_hm'])
print(f"planted NOISE cells through the same pipeline: rho(peak, width) = "
      f"{spearmanr(noise.peak_phase, noise.width_hm).correlation:+.3f}, "
      f"rho(|peak-0.5|, width) = {spearmanr(np.abs(noise.peak_phase - .5), noise.width_hm).correlation:+.3f}, "
      f"median split-half r = {noise.splithalf_r.median():+.3f} (real: {d.splithalf_r.median():+.3f})")

rho(split-half r, width)      = +0.095
rho(split-half r, |peak-0.5|) = +0.128


,n,splithalf_r,width,abs_off_centre,frac_mid_leg
splithalf_r,,,,,
Q1 least reliable,225,0.750,0.433,0.272,0.413
Q2,224,0.891,0.489,0.239,0.429
Q3,224,0.953,0.533,0.261,0.348
Q4 most reliable,225,0.981,0.489,0.361,0.191


,set,all gp-significant,reliable half (split-half r above median)
statistic,group,,
"rho(peak, width)",PFC,0.244,0.386
"rho(|peak-0.5|, width)",PFC,-0.573,-0.542


planted NOISE cells through the same pipeline: rho(peak, width) = +0.058, rho(|peak-0.5|, width) = -0.059, median split-half r = +0.159 (real: +0.933)


### 5.3 Sensitivity to the smoothing width

σ = 3 bins is the primary; σ = 2 and 5 are the sensitivity. Recomputed on the real cells here, and
on the planted cells in section 3.

In [23]:
rows = []
for sg in (2, G.SIGMA, 5):
    tabs = [G.curve_route(caches[rd], sigma=sg)[0] for rd in sorted(caches)]
    Ts = pd.concat(tabs, ignore_index=True).merge(
        T[['recday', 'neuron', 'sel_sig_gp', 'group']], on=['recday', 'neuron'])
    Ss = Ts[Ts.sel_sig_gp]
    Ss = Ss.assign(abs_peak_off_centre=np.abs(Ss.peak_phase - .5))
    for x, key in (('peak_phase', 'rho(peak, width)'), ('abs_peak_off_centre', 'rho(|peak-0.5|, width)')):
        rep = G.rho_report(Ss, x, 'width_hm', groups=['PFC'])
        rows.append({'sigma': sg, 'statistic': key,
                     **{r['group']: round(r['rho_mean_over_mice'], 3) for _, r in rep.iterrows()},
                     'median width': round(float(Ss.width_hm.median()), 3),
                     'not reproduced': round(float(Ss.peak_not_reproduced.mean()), 3)})
display(pd.DataFrame(rows))

,sigma,statistic,PFC,median width,not reproduced
0,2,"rho(peak, width)",0.203,0.467,0.041
1,2,"rho(|peak-0.5|, width)",-0.552,0.467,0.041
2,3,"rho(peak, width)",0.244,0.489,0.034
3,3,"rho(|peak-0.5|, width)",-0.573,0.489,0.034
4,5,"rho(peak, width)",0.288,0.500,0.022
5,5,"rho(|peak-0.5|, width)",-0.548,0.500,0.022


## 6. The β route, and its agreement with the curves

Peak and width read off the ten-bin β profile of the selecting arm and of gp-only/30. Where the two
routes disagree the covariates (place, speed, time-from-reward) are doing the work — that is a
result, not a check.

In [24]:
print('curve route vs beta route (selecting arm):', G.agreement(T))
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4))
axes[0].scatter(S.peak_phase, S.sel_beta_peak_phase, s=4, alpha=.35, color=G.C_NEUTRAL,
                edgecolor='none', rasterized=True)
axes[0].plot([0, 1], [0, 1], color=G.C_STONE, lw=.6, ls='--')
axes[0].set_xlabel('curve peak phase'); axes[0].set_ylabel(r'$\beta$ peak phase (uniform/30)')
axes[1].scatter(S.width_hm, S.sel_beta_width, s=4, alpha=.35, color=G.C_NEUTRAL, edgecolor='none', rasterized=True)
axes[1].plot([0, 1], [0, 1], color=G.C_STONE, lw=.6, ls='--')
axes[1].set_xlabel('curve half-max width'); axes[1].set_ylabel(r'$\beta$ half-max width')
axes[2].scatter(S.sel_beta_peak_phase, S.sel_beta_width, s=4, alpha=.35, color=G.C_NEUTRAL,
                edgecolor='none', rasterized=True)
axes[2].set_xlabel(r'$\beta$ peak phase'); axes[2].set_ylabel(r'$\beta$ width')
fig.tight_layout(); G._save(fig, f'{FIGDIR}/pfc_beta_vs_curve.pdf'); plt.show()
rows = []
for x, key in (('sel_beta_peak_phase', 'rho(beta peak, beta width)'),):
    for arm, pre in (('uniform/30', 'sel_'), ('gp-only/30', 'gpo_')):
        d = T[T[f'{pre}sig_gp']].dropna(subset=[f'{pre}beta_width'])
        d = d.assign(bpk=d[f'{pre}beta_peak_phase'], bw=d[f'{pre}beta_width'],
                     babs=np.abs(d[f'{pre}beta_peak_phase'] - .5))
        rep = G.rho_report(d, 'bpk', 'bw', groups=['PFC'])
        rep2 = G.rho_report(d, 'babs', 'bw', groups=['PFC'])
        for r1, r2 in zip(rep.itertuples(), rep2.itertuples()):
            rows.append({'arm': arm, 'group': r1.group, 'n_mice': r1.n_mice, 'n_units': r1.n_units,
                         'rho(peak,width)': round(r1.rho_mean_over_mice, 3),
                         'rho(|peak-.5|,width)': round(r2.rho_mean_over_mice, 3)})
display(pd.DataFrame(rows))

curve route vs beta route (selecting arm): {'n': 898, 'circ_corr_peaks': 0.2438152711695925, 'spearman_widths': 0.6686865847968302, 'peak_within_1_tenth': 0.7282850779510023, 'median_peak_dist_tenths': 0.4444444444444444}


,arm,group,n_mice,n_units,"rho(peak,width)","rho(|peak-.5|,width)"
0,uniform/30,PFC,7,924,0.267,-0.657
1,gp-only/30,PFC,7,1012,0.328,-0.598


## 7. Per-state peaks and leg duration — the second discriminator

The four legs have different tower distances and so different typical durations. A time cell's
phase peak shifts with the state's median duration; a phase cell's does not. On the planted
populations the per-state circular SD of the peak is an order of magnitude larger for time cells
(synthetic control 4).

In [25]:
rows = []
for rd, c in caches.items():
    pbs = G.peaks_by_state(c['curves'], c['leg_state'], c['leg_duration_s'], sigma=G.SIGMA)
    if len(pbs) < 3:
        continue
    states = sorted(pbs)
    D = np.array([pbs[s]['median_duration_s'] for s in states])
    Pk = np.stack([pbs[s]['peak_phase'] for s in states], axis=1)
    sel = T[(T.recday == rd)].set_index('neuron')
    for i in range(Pk.shape[0]):
        if i not in sel.index or not sel.loc[i, 'sel_sig_gp']:
            continue
        th = 2 * np.pi * Pk[i]
        R = np.abs(np.mean(np.exp(1j * th)))
        rows.append({'recday': rd, 'mouse': rd.split('_')[0], 'neuron': i,
                     'group': sel.loc[i, 'group'],
                     'state_peak_circSD': float(np.sqrt(-2 * np.log(max(R, 1e-12))) / (2 * np.pi)),
                     'rho_peak_vs_duration': float(spearmanr(D, Pk[i]).correlation)
                     if len(set(np.round(D, 3))) > 2 else np.nan,
                     'duration_spread_s': float(D.max() - D.min())})
PS = pd.DataFrame(rows)
sim_sd = {}
for kind in ('tfr', 'ttr', 'phase'):
    sub = sim[sim.kind == kind]
    cols = [c for c in sub.columns if c.startswith('peak_state')]
    th = 2 * np.pi * sub[cols].to_numpy(float)
    R = np.abs(np.nanmean(np.exp(1j * th), axis=1))
    sim_sd[kind] = float(np.nanmedian(np.sqrt(-2 * np.log(np.clip(R, 1e-12, 1))) / (2 * np.pi)))
print('planted per-state peak circular SD (fraction of leg), median:', {k: round(v, 4) for k, v in sim_sd.items()})
display(PS.groupby('group')[['state_peak_circSD', 'rho_peak_vs_duration', 'duration_spread_s']]
        .agg(['count', 'median']).round(3))

/tmp/ipykernel_3820080/3029220260.py:18: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  'rho_peak_vs_duration': float(spearmanr(D, Pk[i]).correlation)


/tmp/ipykernel_3820080/3029220260.py:17: RuntimeWarning: invalid value encountered in sqrt
  'state_peak_circSD': float(np.sqrt(-2 * np.log(max(R, 1e-12))) / (2 * np.pi)),


planted per-state peak circular SD (fraction of leg), median: {'tfr': 0.0694, 'ttr': 0.07, 'phase': 0.0182}


state_peak_circSD        rho_peak_vs_duration        duration_spread_s  \
                  count median                count median             count   
group                                                                          
PFC                 923  0.088                  916   -0.2               930   

              
      median  
group         
PFC    2.162

## 8. The per-region figure guide

One markdown file per region, written from this executed notebook: what every figure shows, which
cells are in it, the unit of inference, what it is **not** evidence for, and that region's own
numbers. Regenerating it here means it cannot drift from the figures.

In [26]:
REPORT_DIR = os.path.join(REPO, 'docs', 'figures',
                          'gp_tuning_width_pfc_abgate')
note = ('' if AB_GATE is None else
        f'> **Robustness variant.** Every figure and number here comes from the run with the A–B '
        f'peak-disagreement gate at {AB_GATE} bins of the 90-bin leg axis (a third of a leg, not '
        f'{AB_GATE}°): cells whose odd-leg and even-leg peaks differ by more than that are '
        f'excluded, the β heatmaps are restricted to the same neurons, and the planted '
        f'populations are gated identically. The gate removes wide cells, so it selects on '
        f'something correlated with the outcome — the ungated run in '
        f'`docs/figures/gp_tuning_width_pfc/` is the primary one.\n>\n'
        f'> One figure here is necessarily identical to the primary run: '
        f'`pfc_width_calibration.pdf` plots only planted *phase* cells, and the '
        f'gate excludes none of them at any threshold, so it cannot change.')
written = G.write_region_reports(S, sim, REPORT_DIR, prefix='pfc',
                                 groups=['PFC'], colors=COLORS,
                                 extra_note=note)
for p in written:
    print(' ', os.path.relpath(p, REPO))

  docs/figures/gp_tuning_width_pfc_abgate/PFC.md
  docs/figures/gp_tuning_width_pfc_abgate/_index.md


## 9. Records

In [27]:
print('written', time.strftime('%Y-%m-%d %H:%M:%S'))
print('caches   ', G.CACHE_DIR)
print('figures  ', FIGDIR)
print('synthetics', G.SYNTH_OUT, '->', sum(v['pass'] for v in SYN['results'].values()),
      'of', len(SYN['results']), 'controls pass')
print('arms     ', list(arms))
print(f'{len(T)} neurons, {T.sel_sig_gp.sum()} gp-significant (uniform/30), '
      f'{np.isfinite(S.width_hm).sum()} with a defined width')

written 2026-09-09 08:17:27
caches    /ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data/processed_data/gp_leg_curves
figures   /ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data/figures/gp_tuning_width_abgate
synthetics /ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data/processed_data/gp_tuning_width_synthetics.pkl -> 8 of 8 controls pass
arms      ['core_progress_time__matched_250ms_decile_cap30s_tfrD10b', 'core_progress_time__matched_250ms_decile_cap30s_tfrU10b', 'core_progress_time__matched_250ms_decile_cap60s_tfrD10b', 'core_progress_time__matched_250ms_decile_cap60s_tfrU10b', 'core_progress_only__matched_250ms_decile_cap30s', 'core_progress_only__matched_250ms_decile_cap60s']
1173 neurons, 930 gp-significant (uniform/30), 898 with a defined width
